In [1]:
import glob, os, shutil
src = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/VERSION", recursive=True)
       if open(p).read().strip() == "v8.6"]
assert src, "v8.6 code not found: attach the dataset with unlearning_tofu_experiment_v8_6.zip"
dst = "/kaggle/working/unlearning-project"
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src[0], dst); os.chdir(dst)
for d in ("results/v6", "logs/v6"):
    shutil.rmtree(d, ignore_errors=True)        # never carry results of an attached notebook into this one
!pip install -q rouge-score bitsandbytes peft
!pip uninstall -y -q torchao
os.makedirs("results/v6", exist_ok=True)
!python env_capture.py > results/v6/environment_run25b1.txt
!nvidia-smi -L
print("code version:", open("VERSION").read().strip())


GPU 0: Tesla T4 (UUID: GPU-b66a9a57-3e97-aee1-912b-d04dc2641a25)
GPU 1: Tesla T4 (UUID: GPU-4507a2c2-9cd3-1f96-8dde-bf4496869c10)
code version: v8.6


In [2]:
# part b needs the output of the run25a1 notebook (Add input -> Notebook output)
import glob
a = glob.glob("/kaggle/input/**/run25_ckpt/A_s1/lora_chain.json", recursive=True)
o = glob.glob("/kaggle/input/**/run25_s1/O_s1_raw.json", recursive=True)
print("Model A adapter:", a[:1]); print("oracle results:", o[:1])
assert a and o, "attach the output of the run25a1 notebook first"


Model A adapter: ['/kaggle/input/notebooks/meerabfatima67/run25a1-ipynb/run25_ckpt/A_s1/lora_chain.json']
oracle results: ['/kaggle/input/notebooks/meerabfatima67/run25a1-ipynb/unlearning-project/results/v6/run25_s1/O_s1_raw.json']


In [3]:
%cd /kaggle/working/unlearning-project
!CKPT_DIR=/tmp/ckpt RUN25_KEEP=/kaggle/working/run25_ckpt python experiments_v6.py run25b1


/kaggle/working
[15:33:13] === v8.6 RUN 25b1  GPUs=2  model=Qwen/Qwen2.5-3B-Instruct (LoRA)  replicate 1
[15:33:14] START  train_P10_s1  (GPU 0)
[16:05:52] DONE   train_P10_s1  (32.6 min)
[16:05:52] START  train_r25_anchor_p10_forget10_lr0.0003_s1  (GPU 1)
[16:05:52] START  train_r25_npo_rt_forget10_lr0.0001_s1  (GPU 0)
[18:16:58] DONE   train_r25_npo_rt_forget10_lr0.0001_s1  (131.1 min)
[18:16:58] START  eval_run25_s1_npo_rt_lr0.0001_ep10_s1  (GPU 0)
[18:40:00] DONE   train_r25_anchor_p10_forget10_lr0.0003_s1  (154.1 min)
[18:40:00] START  eval_run25_s1_anchor_p10_lr0.0003_ep10_s1  (GPU 1)
[19:07:24] DONE   eval_run25_s1_npo_rt_lr0.0001_ep10_s1  (50.4 min)
[19:07:24] START  eval_run25_s1_npo_rt_lr0.0001_ep5_s1  (GPU 0)
[19:25:52] DONE   eval_run25_s1_anchor_p10_lr0.0003_ep10_s1  (45.9 min)
[19:25:52] START  eval_run25_s1_anchor_p10_lr0.0003_ep5_s1  (GPU 1)
[19:53:20] DONE   eval_run25_s1_npo_rt_lr0.0001_ep5_s1  (45.9 min)
[20:11:37] DONE   eval_run25_s1_anchor_p10_lr0.0003_ep5_s1  (45

In [4]:
%cd /kaggle/working/unlearning-project
!zip -qr /kaggle/working/results_v8_6_run25b1.zip results/v6/run25_s1 results/v6/environment_run25b1.txt logs/v6
!ls -la /kaggle/working


/kaggle/working
total 560
drwxr-xr-x 3 root root   4096 Nov 17 20:11 .
drwxr-xr-x 8 root root   4096 Nov 17 15:32 ..
-rw-r--r-- 1 root root   8269 Nov 17 20:11 __notebook__.ipynb
-rw-r--r-- 1 root root 547635 Nov 17 20:11 results_v8_6_run25b1.zip
drwxrwxrwx 8 root root   4096 Nov 17 15:32 unlearning-project
